In [1]:
import sqlite3
# print the drive paths of each notegroup
DB_PATH = "DB/oedb_baseline.db"
service_account_file = "config/service_account_key.json"

from oral_notes.s1_extract.doc_loader import GoogleDriveLoader
from oral_notes.s1_extract.text_extractor import TextExtractor

file_loader = GoogleDriveLoader(service_account_file)
extractor = TextExtractor()

with sqlite3.connect(DB_PATH) as conn:
    cursor = conn.cursor()

    for notegroup_id in range(1, 24):
        cursor.execute("""
            SELECT project_name, phase, note_url_QA, note_url_PARTICIPANT
            FROM notegroups
            WHERE notegroupID = ?
        """, (notegroup_id,))
        row = cursor.fetchone()

        if row is None:
            print(f"[{notegroup_id}] No notegroup found — skipping")
            continue

        project_name, phase, note_url_qa, note_url_participant = row
        all_drive_paths = {}

        for label, url in [("QA", note_url_qa), ("PARTICIPANT", note_url_participant)]:
            if not url:
                continue
            try:
                result = file_loader.load(url)
                all_drive_paths[label] = result['drive_path']
            except Exception as e:
                print(f"[{notegroup_id}] {label} load failed: {e}")

        combined_drive_paths = "|".join(all_drive_paths.values())
        print(f"[{notegroup_id}] {project_name} / {phase}: {combined_drive_paths}")

Loading: Rama - Note-taking 3 (1).docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)
Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Rama - Note-taking 3 (1).docx
Loading: Aanwezig sessie 1 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Aanwezig sessie 1
[1] Helmond de Peel (2025) / 1: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Rama - Note-taking 3 (1).docx|01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Aanwezig sessie 1
Loading: Iyad - Note-taking 3.12 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. 